# Laboratorio de métodos de búsqueda

|                |   |
:----------------|---|
| **Nombre**     | Cecilia Quiroz Hidalgo  |
| **Fecha**      | 28/10/2026  |
| **Expediente** | 757942  | 

Modelos como regresión logística, máquinas de vectores de soporte, y preceptrones multicapa tienen una serie de argumentos que necesitan valores dados antes de poder usarse para ajustarse a un dataset. Estos argumentos se llaman hiperparámetros.

¿Cuál es la diferencia entre un parámetro y un hiperparámetro?

- Un parámetro es un valor que el modelo aprende durante el entrenamiento a partir de los datos, mientras que un hiperparámetro es un valor que definimos antes del entrenamiento. 

Este laboratorio se centra en métodos de búsqueda para encontrar los mejores hiperparámetros para un modelo dado. Imagina que estás jugando batalla naval:

<center><img src="https://cdn11.bigcommerce.com/s-nehqfptb/images/stencil/670x993/products/40096/100569/battleship-game__73281.1768238533.jpg?c=2"></center>

Una forma de jugar podría ser ir de cuadro en cuadro hasta probar todo el tablero. El problema aquí es que se juega a contrarreloj, debido a que tu oponente puede hundir tus barcos antes de que pruebes todo el tablero.

Si no se tiene la preocupación/limitante de tiempo, probar todas las combinaciones es una opción:

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

Entra a la documentación de `GridSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros).

GridSearchCV busca los mejores hiperparámetros de un modelo probando combinaciones que nosotros mismos le indicamos. Los parámetros que recibe son:

**Entradas:**

- estimator: el modelo a ajustar
- param_grid: diccionario con cada hiperparámetro y la lista de valores a probar
- scoring: métrica para evaluar
- n_jobs: número de procesos en paralelo 
- refit: si es True, al final vuelve a entrenar el modelo con los mejores hiperparámetros usando todos los datos
- cv: número de folds de la validación cruzada
- pre_dispatch: cuántos trabajos se mandan a la vez cuando se trabaja en paralelo
- error_score: valor que se asigna si un ajuste falla
- return_train_score: si es True, también guarda el score en entrenamiento

**Salidas:**

- best_params_: la mejor combinación de hiperparámetros.
- best_score_: el score promedio de esa combinación.
- best_estimator_: el modelo ya entrenado con los mejores hiperparámetros.
- cv_results_: resultados de todas las combinaciones probadas.

Otra forma de "buscar" podría ser tirar al azar y ver qué pasa. Esto sería otra forma de utilizar la fuerza bruta para encontrar información. 

https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html

Entra a la documentación de `RandomizedSearchCV` de `sklearn` y describe cómo funciona este objeto (entradas, salidas, parámetros). 

RandomizedSearchCV funciona igual que GridSearchCV, pero no prueba todas las combinaciones, sino que elige al azar un número fijo de ellas. Por eso es más rápido cuando hay muchas combinaciones posibles.

**Entradas**

- estimator: el modelo a ajustar
- param_distributions: diccionario con los hiperparámetros y los valores (o distribuciones) de donde se van a escoger al azar
- n_iter: cuántas combinaciones se van a probar (por defecto 10)
- random_state: semilla para que el resultado se pueda repetir
- scoring, n_jobs, refit, cv, pre_dispatch, error_score, return_train_score: igual que en GridSearchCV

**Salidas:** 

- Las mismas que GridSearchCV (best_params_, best_score_, best_estimator_, cv_results_), pero solo con las n_iter combinaciones que se probaron.

**Nota:** RandomizedSearchCV prueba puramente al azar. Si encontrara en uno de sus tiros que se atinó a un barco, no cambia el algoritmo de búsqueda para seguir buscando en esa zona. Para un proceso de búsqueda más refinado necesitaríamos un algoritmo que cambie su forma de decidir a partir de información anterior.

Revisa la documentación de Optuna (https://optuna.org/). Instala la librería y escribe una guía de cómo se debe utlizar para un dataset dado.

Optuna es una librería para buscar hiperparámetros que aprende de los intentos anteriores, si una zona dio buenos resultados, los siguientes intentos se concentran ahí.

**Guía de uso para un dataset:**

1. Instalar: `pip install optuna`
2. Preparar los datos: separar `X` (variables) y `y` (lo que se quiere predecir).
3. Definir una función objetivo: recibe un `trial`, le pide los hiperparámetros con `trial.suggest_int(...)`, `trial.suggest_float(...)` o `trial.suggest_categorical(...)`, entrena el modelo y regresa el score.
4. Crear el estudio: `estudio = optuna.create_study(direction="maximize")`
5. Optimizar: `estudio.optimize(objetivo, n_trials=20)`
6. Ver resultados: `estudio.best_params` y `estudio.best_value`.

### Ejercicio

Utiliza el dataset `College.csv` para intentar predecir si una universidad es privada a partir del resto de los factores.

Utiliza un perceptrón multicapa con hiperparámetros:
- Número de capas intermedias --> {2, 3, 4}
- Número de neuronas en capa 1 --> [5, 10]
- Número de neuronas en capa 2 --> [10, 20]
- Número de neuronas en capa 3 (si existe) --> [5, 20]
- Número de neuronas en capa 4 (si existe) --> [5, 10]

Utiliza k-folds cross-validation con $k=10$; optimiza el ROC-AUC de test.

Compara los 3 diferentes métodos de búsqueda descritos en este laboratorio.

In [4]:
!pip install optuna

import pandas as pd
import time
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
import optuna

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: C:\Users\ceciq\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
df = pd.read_csv("College.csv", index_col=0)

In [9]:
X = df.drop(columns="Private")
y = df["Private"].map({"Yes": 1, "No": 0})

In [10]:
X = StandardScaler().fit_transform(X)

#### 1. GridSearchCV

In [11]:
capas = []
for c1 in [5, 10]:
    for c2 in [10, 20]:
        capas.append((c1, c2))
        for c3 in [5, 20]:
            capas.append((c1, c2, c3))
            for c4 in [5, 10]:
                capas.append((c1, c2, c3, c4))

print("Combinaciones:", len(capas))

grid = GridSearchCV(MLPClassifier(max_iter=1000, random_state=0),
                    {"hidden_layer_sizes": capas},
                    scoring="roc_auc", cv=10, n_jobs=-1)

grid.fit(X, y)

print("Mejores capas:", grid.best_params_)
print("ROC-AUC:", grid.best_score_)

Combinaciones: 28
Mejores capas: {'hidden_layer_sizes': (10, 10)}
ROC-AUC: 0.969023546420163


#### 2. RandomizedSearchCV

In [12]:
capas_todas = []
for c1 in range(5, 11):
    for c2 in range(10, 21):
        capas_todas.append((c1, c2))
        for c3 in range(5, 21):
            capas_todas.append((c1, c2, c3))
            for c4 in range(5, 11):
                capas_todas.append((c1, c2, c3, c4))

print("Combinaciones posibles:", len(capas_todas))

aleatorio = RandomizedSearchCV(MLPClassifier(max_iter=1000, random_state=0),
                               {"hidden_layer_sizes": capas_todas},
                               n_iter=28, scoring="roc_auc", cv=10,
                               n_jobs=-1, random_state=0)

aleatorio.fit(X, y)

print("Mejores capas:", aleatorio.best_params_)
print("ROC-AUC:", aleatorio.best_score_)

Combinaciones posibles: 7458
Mejores capas: {'hidden_layer_sizes': (7, 11)}
ROC-AUC: 0.9698340141479239


### 3. Optuna

In [13]:
def objetivo(trial):
    n_capas = trial.suggest_int("n_capas", 2, 4)

    capas = [trial.suggest_int("capa1", 5, 10),
             trial.suggest_int("capa2", 10, 20)]
    if n_capas >= 3:
        capas.append(trial.suggest_int("capa3", 5, 20))
    if n_capas == 4:
        capas.append(trial.suggest_int("capa4", 5, 10))

    modelo = MLPClassifier(hidden_layer_sizes=capas, max_iter=1000, random_state=0)
    return cross_val_score(modelo, X, y, scoring="roc_auc", cv=10, n_jobs=-1).mean()

estudio = optuna.create_study(direction="maximize")

estudio.optimize(objetivo, n_trials=28)

print("Mejores capas:", estudio.best_params)
print("ROC-AUC:", estudio.best_value)

[I 2026-10-28 12:33:51,736] A new study created in memory with name: no-name-29c218a3-ff60-43e9-a74a-b54b9bfd810d
[I 2026-10-28 12:33:54,829] Trial 0 finished with value: 0.9580762647961896 and parameters: {'n_capas': 2, 'capa1': 6, 'capa2': 16}. Best is trial 0 with value: 0.9580762647961896.
[I 2026-10-28 12:33:57,904] Trial 1 finished with value: 0.9607292717725049 and parameters: {'n_capas': 2, 'capa1': 9, 'capa2': 20}. Best is trial 1 with value: 0.9607292717725049.
[I 2026-10-28 12:34:01,034] Trial 2 finished with value: 0.9620091896407684 and parameters: {'n_capas': 2, 'capa1': 6, 'capa2': 11}. Best is trial 2 with value: 0.9620091896407684.
[I 2026-10-28 12:34:04,053] Trial 3 finished with value: 0.9662267817812931 and parameters: {'n_capas': 2, 'capa1': 10, 'capa2': 14}. Best is trial 3 with value: 0.9662267817812931.
[I 2026-10-28 12:34:09,493] Trial 4 finished with value: 0.926387871735616 and parameters: {'n_capas': 4, 'capa1': 5, 'capa2': 16, 'capa3': 6, 'capa4': 10}. Best

Mejores capas: {'n_capas': 2, 'capa1': 7, 'capa2': 13}
ROC-AUC: 0.9688350340136056


### Comparación

In [14]:
comparacion = pd.DataFrame({
    "Método": ["GridSearchCV", "RandomizedSearchCV", "Optuna"],
    "ROC-AUC": [grid.best_score_, aleatorio.best_score_, estudio.best_value],
})
comparacion

,Método,ROC-AUC
0,GridSearchCV,0.969024
1,RandomizedSearchCV,0.969834
2,Optuna,0.968835


### Conclusiones

En general todos los métodos obtuvieron un buen puntaje y no hubo mucha diferencia entre ellos, así que la cantidad de capas y neuronas no tuvo mucho efecto sobre ningún método, ya que casi todas las combinaciones dieron buenos resultados. 

- GridSearchCV tuvo buenos resultados probando solo algunas combinaciones.
- RandomizedSearchCV tuvo el mejor resultado, aunque por una diferencia muy pequeña.
- Optuna no tuvo ventaja porque los resultados eran muy similares. Funciona mejor cuando hay más parámetros.